# Advanced `collections.Counter` — Problems with Complete Solutions

This notebook expands a lesson on Python's `collections.Counter` into a larger, advanced practice set.

## Topics covered

- Constructing counters from iterables, mappings, and keyword arguments
- Frequency distributions and robust text normalization
- Missing-key behavior
- `most_common`
- `elements`
- `update` and `subtract`
- Counter arithmetic: `+`, `-`, `&`, `|`
- Unary `+` and unary `-`
- Multiset reasoning
- Orders, refunds, inventory, ledgers, and reconciliation
- Deterministic top-k ranking
- Sliding-window algorithms
- Minimum-window coverage
- Sparse-vector similarity
- Streaming/batched counting
- Comparing `Counter` solutions with plain dictionaries
- Testing, edge cases, and complexity

Every major problem includes a **complete solution** and **assertions/tests**.

## Best-practice goals

Throughout the notebook, prefer:

1. Small functions with clear inputs/outputs.
2. Type hints and docstrings for reusable logic.
3. `Counter` operations whose semantics match the problem.
4. `subtract()` when signed/negative results matter.
5. Counter `-` only when non-positive counts should be discarded.
6. Direct weighted aggregation instead of expanding large quantities into repeated elements.
7. Explicit deterministic sorting when tie-breaking rules matter.
8. Assertions for correctness and edge cases.
9. Memory-aware designs for large streams and inventories.
10. Clear separation between **counting**, **normalization**, and **reporting**.

In [1]:
from collections import Counter, defaultdict
from collections.abc import Iterable, Mapping
from itertools import chain, repeat
from math import sqrt
from typing import Hashable, TypeVar
import random
import re
import unicodedata

T = TypeVar("T", bound=Hashable)

# Part I — Core behavior refresher

## Example 1 — `defaultdict(int)` versus `Counter`

Both can count items. `Counter` is specialized for counting and adds methods and multiset operations.

In [2]:
sentence = "the quick brown fox jumps over the lazy dog"

dd = defaultdict(int)
for ch in sentence:
    dd[ch] += 1

counter = Counter(sentence)

assert dict(dd) == dict(counter)
counter.most_common(5)

[(' ', 8), ('o', 4), ('e', 3), ('t', 2), ('h', 2)]

## Example 2 — Construction styles

In [3]:
from_iterable = Counter("mississippi")
from_mapping = Counter({"red": 3, "blue": 2})
from_keywords = Counter(red=3, blue=2)

assert from_mapping == from_keywords

print("iterable:", from_iterable)
print("mapping :", from_mapping)
print("kwargs  :", from_keywords)

iterable: Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})
mapping : Counter({'red': 3, 'blue': 2})
kwargs  : Counter({'red': 3, 'blue': 2})


## Example 3 — Missing keys return zero

A missing key behaves like a count of zero.

In [4]:
c = Counter(a=2)

assert c["missing"] == 0
assert "missing" not in c

c["missing"] += 1
assert c["missing"] == 1

c

Counter({'a': 2, 'missing': 1})

## Example 4 — `most_common`

In [5]:
colors = Counter(
    ["red", "blue", "red", "green", "red", "blue", "yellow", "green", "red"]
)

print(colors.most_common())
print(colors.most_common(2))

[('red', 4), ('blue', 2), ('green', 2), ('yellow', 1)]
[('red', 4), ('blue', 2)]


## Example 5 — `elements()`

`elements()` repeats each key according to its positive integer count. Zero and negative counts are ignored.

In [6]:
c = Counter(a=3, b=2, c=0, d=-4)

expanded = list(c.elements())

assert Counter(expanded) == Counter(a=3, b=2)
expanded

['a', 'a', 'a', 'b', 'b']

## Example 6 — `update()` is additive

In [7]:
c = Counter("abb")
c.update("bcc")

assert c == Counter(a=1, b=3, c=2)
c

Counter({'b': 3, 'c': 2, 'a': 1})

## Example 7 — `subtract()` preserves signed counts

In [8]:
c = Counter(a=5, b=2)
c.subtract(Counter(a=7, b=1, c=3))

assert c == Counter(a=-2, b=1, c=-3)
c

Counter({'b': 1, 'a': -2, 'c': -3})

## Example 8 — Counter arithmetic

For two counters `a` and `b`:

- `a + b`: add counts, keeping positive results
- `a - b`: subtract counts, keeping positive results
- `a & b`: elementwise minimum, keeping positive results
- `a | b`: elementwise maximum, keeping positive results

In [9]:
a = Counter(apples=5, bananas=2, pears=1)
b = Counter(apples=2, bananas=4, oranges=3)

print("a + b:", a + b)
print("a - b:", a - b)
print("a & b:", a & b)
print("a | b:", a | b)

a + b: Counter({'apples': 7, 'bananas': 6, 'oranges': 3, 'pears': 1})
a - b: Counter({'apples': 3, 'pears': 1})
a & b: Counter({'apples': 2, 'bananas': 2})
a | b: Counter({'apples': 5, 'bananas': 4, 'oranges': 3, 'pears': 1})


## Example 9 — Unary cleanup operators

Unary `+` keeps only positive counts.

Unary `-` flips signs and then keeps only positive counts.

In [10]:
balance = Counter(in_stock=10, exact=0, backordered=-4)

positive = +balance
negative_magnitudes = -balance

assert positive == Counter(in_stock=10)
assert negative_magnitudes == Counter(backordered=4)

positive, negative_magnitudes

(Counter({'in_stock': 10}), Counter({'backordered': 4}))

## Example 10 — A compatibility-friendly total helper

Modern Python versions provide `Counter.total()`. `sum(counter.values())` is a simple compatibility pattern.

In [11]:
def counter_total(c: Counter) -> int | float:
    """Return the arithmetic total of all counts, including negative counts."""
    return sum(c.values())

c = Counter(a=4, b=3, c=-2)
assert counter_total(c) == 5
counter_total(c)

5

# Part II — Advanced problems with solutions

## Problem 1 — Character-frequency report with normalization

Write `character_report(text, top_n=5)` that:

- ignores whitespace,
- treats upper/lower case as equal,
- counts all remaining characters,
- returns `(counter, top_n_items)`.

### Example

`"Data DATA!"` should count `d` twice, `a` four times, `t` twice, and `!` once.

In [12]:
def character_report(text: str, top_n: int = 5) -> tuple[Counter[str], list[tuple[str, int]]]:
    """Count non-whitespace characters case-insensitively."""
    if top_n < 0:
        raise ValueError("top_n must be >= 0")

    normalized = (ch.casefold() for ch in text if not ch.isspace())
    counts = Counter(normalized)
    return counts, counts.most_common(top_n)


counts, top = character_report("Data DATA!", top_n=4)

assert counts == Counter({"a": 4, "d": 2, "t": 2, "!": 1})
assert top[0] == ("a", 4)

counts, top

(Counter({'a': 4, 'd': 2, 't': 2, '!': 1}),
 [('a', 4), ('d', 2), ('t', 2), ('!', 1)])

## Problem 2 — Robust word frequencies

The basic `re.split(r"\W", text)` approach can create empty strings. Build a safer tokenizer that:

- uses `casefold()`,
- extracts words instead of splitting on every non-word character,
- allows apostrophes inside words,
- returns a `Counter`.

Then return the top `n` words using deterministic tie-breaking:
1. larger count first,
2. alphabetically for ties.

In [13]:
WORD_RE = re.compile(r"[^\W_]+(?:'[^\W_]+)*", re.UNICODE)


def tokenize_words(text: str) -> list[str]:
    """Extract normalized words, allowing internal apostrophes."""
    return [match.group(0).casefold() for match in WORD_RE.finditer(text)]


def word_frequencies(text: str) -> Counter[str]:
    """Return normalized word frequencies."""
    return Counter(tokenize_words(text))


def deterministic_top_words(text: str, n: int) -> list[tuple[str, int]]:
    """Return top words by count descending, then word ascending."""
    if n < 0:
        raise ValueError("n must be >= 0")

    counts = word_frequencies(text)
    return sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:n]


sample = "Python, python! Counter's counters; COUNTER'S code. Code?"
counts = word_frequencies(sample)
top = deterministic_top_words(sample, 4)

assert counts["python"] == 2
assert counts["counter's"] == 2
assert counts["code"] == 2
assert "" not in counts

counts, top

(Counter({'python': 2, "counter's": 2, 'code': 2, 'counters': 1}),
 [('code', 2), ("counter's", 2), ('python', 2), ('counters', 1)])

## Problem 3 — Weighted records without materializing repeated elements

You receive records like:

```python
[
    ("battery", 4),
    ("mouse", 2),
    ("battery", 3),
]
```

Build a counter of total quantities.

### Best-practice constraint

Do **not** expand `("battery", 1_000_000)` into one million strings. Aggregate weights directly.

In [14]:
def weighted_counter(records: Iterable[tuple[T, int]]) -> Counter[T]:
    """Aggregate (item, quantity) records without expanding repeated elements."""
    result: Counter[T] = Counter()

    for item, quantity in records:
        if not isinstance(quantity, int):
            raise TypeError("quantity must be an integer")
        result[item] += quantity

    return result


orders = [
    ("battery", 4),
    ("mouse", 2),
    ("battery", 3),
    ("keyboard", 5),
    ("mouse", 4),
]

totals = weighted_counter(orders)

assert totals == Counter(battery=7, mouse=6, keyboard=5)
totals

Counter({'battery': 7, 'mouse': 6, 'keyboard': 5})

### Why this is preferable to `repeat()` + `chain`

`Counter(chain.from_iterable(repeat(item, qty) ...))` can be elegant for small demonstrations, but direct weighted aggregation performs work proportional to the **number of records**, not the total expanded quantity.

In [15]:
tiny_records = [("a", 3), ("b", 2)]

expanded_version = Counter(
    chain.from_iterable(repeat(item, qty) for item, qty in tiny_records)
)
direct_version = weighted_counter(tiny_records)

assert expanded_version == direct_version
expanded_version

Counter({'a': 3, 'b': 2})

## Problem 4 — Orders minus refunds: choose the correct subtraction semantics

Implement two functions:

1. `net_positive_sales`: net sales where non-positive results are discarded.
2. `signed_net_sales`: preserve negative results so over-refunds remain visible.

Use the correct `Counter` operation for each.

In [16]:
def net_positive_sales(
    orders: Iterable[tuple[str, int]],
    refunds: Iterable[tuple[str, int]],
) -> Counter[str]:
    """Return positive net units only."""
    sold = weighted_counter(orders)
    returned = weighted_counter(refunds)
    return sold - returned


def signed_net_sales(
    orders: Iterable[tuple[str, int]],
    refunds: Iterable[tuple[str, int]],
) -> Counter[str]:
    """Return signed net units, preserving zeros/negatives in the Counter."""
    net = weighted_counter(orders)
    net.subtract(weighted_counter(refunds))
    return net


orders = [("battery", 10), ("mouse", 3)]
refunds = [("battery", 4), ("mouse", 5), ("cable", 2)]

positive = net_positive_sales(orders, refunds)
signed = signed_net_sales(orders, refunds)

assert positive == Counter(battery=6)
assert signed == Counter(battery=6, mouse=-2, cable=-2)

positive, signed

(Counter({'battery': 6}), Counter({'battery': 6, 'mouse': -2, 'cable': -2}))

## Problem 5 — Inventory reconciliation

Given:

- `expected`: inventory from the system,
- `physical`: inventory counted in the warehouse,

return:

- `overages`: physical count above expected,
- `shortages`: expected count above physical,
- `signed_delta`: physical - expected.

Use Counter operations to keep the meaning explicit.

In [17]:
def reconcile_inventory(
    expected: Mapping[str, int],
    physical: Mapping[str, int],
) -> tuple[Counter[str], Counter[str], Counter[str]]:
    """Return (overages, shortages, signed_delta)."""
    expected_c = Counter(expected)
    physical_c = Counter(physical)

    signed_delta = physical_c.copy()
    signed_delta.subtract(expected_c)

    overages = +signed_delta
    shortages = -signed_delta

    return overages, shortages, signed_delta


expected = {"battery": 20, "mouse": 12, "cable": 8}
physical = {"battery": 18, "mouse": 15, "cable": 8, "keyboard": 2}

over, short, delta = reconcile_inventory(expected, physical)

assert over == Counter(mouse=3, keyboard=2)
assert short == Counter(battery=2)
assert delta == Counter(mouse=3, keyboard=2, cable=0, battery=-2)

over, short, delta

(Counter({'mouse': 3, 'keyboard': 2}),
 Counter({'battery': 2}),
 Counter({'mouse': 3, 'keyboard': 2, 'cable': 0, 'battery': -2}))

## Problem 6 — Multiset containment

A recipe/order requires quantities of several items.

Write `can_fulfill(available, required)` that returns `True` only when every **positive** required count is available in sufficient quantity.

Negative or zero requirements should not create a requirement.

In [18]:
def can_fulfill(
    available: Mapping[T, int],
    required: Mapping[T, int],
) -> bool:
    """Return whether available counts cover all positive requirements."""
    available_c = Counter(available)
    required_c = +Counter(required)

    return all(available_c[item] >= qty for item, qty in required_c.items())


assert can_fulfill(
    {"flour": 5, "egg": 6, "milk": 2},
    {"flour": 3, "egg": 4},
)

assert not can_fulfill(
    {"flour": 2, "egg": 6},
    {"flour": 3, "egg": 4},
)

assert can_fulfill(
    {"a": 1},
    {"a": 1, "ignored": 0, "also_ignored": -5},
)

## Problem 7 — Missing quantities for an order

Write `missing_items(available, required)` that returns a Counter showing exactly how much more is needed.

This is a natural multiset subtraction problem.

In [19]:
def missing_items(
    available: Mapping[T, int],
    required: Mapping[T, int],
) -> Counter[T]:
    """Return positive deficits required - available."""
    return (+Counter(required)) - (+Counter(available))


available = {"battery": 3, "cable": 2}
required = {"battery": 5, "cable": 1, "mouse": 2}

missing = missing_items(available, required)

assert missing == Counter(battery=2, mouse=2)
missing

Counter({'battery': 2, 'mouse': 2})

## Problem 8 — Unicode-aware anagram detection

Write an anagram checker that:

- normalizes Unicode using NFKC,
- case-folds,
- keeps only alphanumeric characters,
- uses `Counter`.

This is more robust than simply deleting ASCII spaces.

In [20]:
def normalized_alnum(text: str) -> list[str]:
    """Return NFKC-normalized, casefolded alphanumeric characters."""
    normalized = unicodedata.normalize("NFKC", text).casefold()
    return [ch for ch in normalized if ch.isalnum()]


def are_anagrams(left: str, right: str) -> bool:
    """Return whether two strings are anagrams after normalization."""
    return Counter(normalized_alnum(left)) == Counter(normalized_alnum(right))


assert are_anagrams("Dormitory", "Dirty room!!")
assert are_anagrams("The Eyes", "They see")
assert not are_anagrams("Counter", "Counters")

## Problem 9 — Maximum number of complete kits

A kit requires multiple components. Given available inventory, compute how many **complete kits** can be built.

For example, if one kit requires 2 batteries and 1 cable, and inventory has 9 batteries and 7 cables, the answer is 4.

In [21]:
def max_complete_kits(
    available: Mapping[T, int],
    required_per_kit: Mapping[T, int],
) -> int:
    """Return the maximum number of complete kits that can be built."""
    available_c = Counter(available)
    required_c = +Counter(required_per_kit)

    if not required_c:
        raise ValueError("required_per_kit must contain at least one positive count")

    return min(available_c[item] // qty for item, qty in required_c.items())


assert max_complete_kits(
    {"battery": 9, "cable": 7},
    {"battery": 2, "cable": 1},
) == 4

assert max_complete_kits(
    {"battery": 10, "cable": 0},
    {"battery": 2, "cable": 1},
) == 0

## Problem 10 — Deterministic top-k ranking

`most_common(k)` is excellent when any valid ordering among ties is acceptable.

For reporting systems, you may need an explicit rule.

Write `top_k_deterministic(counter, k)` that sorts by:

1. count descending,
2. `repr(key)` ascending for ties.

This works for heterogeneous hashable keys without requiring keys to be mutually comparable.

In [22]:
def top_k_deterministic(c: Counter[T], k: int) -> list[tuple[T, int]]:
    """Return deterministic top-k results by count desc, repr(key) asc."""
    if k < 0:
        raise ValueError("k must be >= 0")

    return sorted(
        c.items(),
        key=lambda item: (-item[1], repr(item[0])),
    )[:k]


c = Counter({"beta": 4, "alpha": 4, 10: 4, "gamma": 2})
result = top_k_deterministic(c, 3)

assert [count for _, count in result] == [4, 4, 4]
result

[('alpha', 4), ('beta', 4), (10, 4)]

## Problem 11 — Find all anagram windows in a string

Given a text `s` and pattern `p`, return all start indices where the length-`len(p)` substring of `s` is an anagram of `p`.

### Constraint

Use a sliding window so the solution is linear in the length of `s` aside from Counter comparison overhead.

In [23]:
def find_anagram_starts(s: str, p: str) -> list[int]:
    """Return start indices of windows in s that are anagrams of p."""
    if not p or len(p) > len(s):
        return []

    target = Counter(p)
    window = Counter(s[: len(p)])
    result = []

    if window == target:
        result.append(0)

    for right in range(len(p), len(s)):
        incoming = s[right]
        outgoing = s[right - len(p)]

        window[incoming] += 1
        window[outgoing] -= 1

        if window[outgoing] == 0:
            del window[outgoing]

        if window == target:
            result.append(right - len(p) + 1)

    return result


assert find_anagram_starts("cbaebabacd", "abc") == [0, 6]
assert find_anagram_starts("abab", "ab") == [0, 1, 2]
assert find_anagram_starts("abc", "") == []

## Problem 12 — Minimum window covering a multiset

Given a string `s` and string `t`, find the shortest substring of `s` containing all characters of `t` with multiplicity.

Example:

- `s = "ADOBECODEBANC"`
- `t = "ABC"`
- answer: `"BANC"`

Use `Counter` for requirements and a sliding window.

In [24]:
def min_cover_window(s: str, t: str) -> str:
    """Return the shortest substring of s covering all characters in t."""
    if not t or not s or len(t) > len(s):
        return ""

    need = Counter(t)
    have = Counter()

    required_kinds = len(need)
    satisfied_kinds = 0

    left = 0
    best_start = 0
    best_len = float("inf")

    for right, ch in enumerate(s):
        have[ch] += 1

        if ch in need and have[ch] == need[ch]:
            satisfied_kinds += 1

        while satisfied_kinds == required_kinds:
            current_len = right - left + 1

            if current_len < best_len:
                best_start = left
                best_len = current_len

            outgoing = s[left]
            have[outgoing] -= 1

            if outgoing in need and have[outgoing] < need[outgoing]:
                satisfied_kinds -= 1

            left += 1

    if best_len == float("inf"):
        return ""

    return s[best_start : best_start + best_len]


assert min_cover_window("ADOBECODEBANC", "ABC") == "BANC"
assert min_cover_window("a", "aa") == ""
assert min_cover_window("aa", "aa") == "aa"

## Problem 13 — Multiset intersection and union metrics

For two shopping baskets represented by Counters:

- intersection (`&`) gives shared quantities,
- union (`|`) gives maximum quantities.

Implement multiset Jaccard similarity:

\[
J(A,B) = \frac{\sum(A \& B)}{\sum(A \mid B)}
\]

Return `1.0` when both positive multisets are empty.

In [25]:
def multiset_jaccard(a: Mapping[T, int], b: Mapping[T, int]) -> float:
    """Return multiset Jaccard similarity on positive counts."""
    a_c = +Counter(a)
    b_c = +Counter(b)

    intersection = a_c & b_c
    union = a_c | b_c

    union_size = sum(union.values())
    if union_size == 0:
        return 1.0

    return sum(intersection.values()) / union_size


score = multiset_jaccard(
    {"apple": 3, "banana": 1},
    {"apple": 2, "banana": 3},
)

assert score == 3 / 6
assert multiset_jaccard({}, {}) == 1.0
score

0.5

## Problem 14 — Sparse cosine similarity with Counters

Treat Counters as sparse vectors of feature frequencies.

Implement cosine similarity without constructing a dense vector.

In [26]:
def counter_cosine_similarity(
    a: Mapping[T, int | float],
    b: Mapping[T, int | float],
) -> float:
    """Return cosine similarity between sparse count vectors."""
    a_c = Counter(a)
    b_c = Counter(b)

    common_keys = a_c.keys() & b_c.keys()
    dot = sum(a_c[key] * b_c[key] for key in common_keys)

    norm_a = sqrt(sum(value * value for value in a_c.values()))
    norm_b = sqrt(sum(value * value for value in b_c.values()))

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot / (norm_a * norm_b)


doc1 = Counter("data science data".split())
doc2 = Counter("data engineering science".split())

similarity = counter_cosine_similarity(doc1, doc2)

assert 0 < similarity < 1
similarity

0.7745966692414834

## Problem 15 — Merge many shard counters

Imagine log events are counted independently on several workers.

Implement `merge_counters(shards)` without mutating any input Counter.

In [27]:
def merge_counters(shards: Iterable[Counter[T]]) -> Counter[T]:
    """Merge counters additively into a fresh Counter."""
    merged: Counter[T] = Counter()

    for shard in shards:
        merged.update(shard)

    return merged


shards = [
    Counter(ok=100, error=3),
    Counter(ok=120, warning=5),
    Counter(ok=80, error=2),
]

merged = merge_counters(shards)

assert merged == Counter(ok=300, warning=5, error=5)
assert shards[0] == Counter(ok=100, error=3)

merged

Counter({'ok': 300, 'error': 5, 'warning': 5})

### Alternative

For small collections, `sum(shards, Counter())` also works. An explicit loop is often easier to read and extend with validation or logging.

In [28]:
summed = sum(shards, Counter())
assert summed == merged
summed

Counter({'ok': 300, 'error': 5, 'warning': 5})

## Problem 16 — Event ledger with reversals

Events arrive as `(name, delta)` where `delta` may be positive or negative.

Build a signed ledger, then report:

- positive balances,
- negative balances as positive magnitudes,
- zero-balance keys.

This problem demonstrates why deleting non-positive values too early can lose information.

In [29]:
def build_signed_ledger(events: Iterable[tuple[str, int]]) -> Counter[str]:
    """Aggregate signed deltas."""
    ledger: Counter[str] = Counter()

    for name, delta in events:
        ledger[name] += delta

    return ledger


def ledger_report(
    events: Iterable[tuple[str, int]],
) -> tuple[Counter[str], Counter[str], set[str], Counter[str]]:
    """Return positive, negative magnitudes, zero keys, and full signed ledger."""
    ledger = build_signed_ledger(events)
    positive = +ledger
    negative = -ledger
    zero_keys = {key for key, value in ledger.items() if value == 0}

    return positive, negative, zero_keys, ledger


events = [
    ("A", 10),
    ("B", 4),
    ("A", -3),
    ("B", -4),
    ("C", -2),
]

positive, negative, zero_keys, ledger = ledger_report(events)

assert positive == Counter(A=7)
assert negative == Counter(C=2)
assert zero_keys == {"B"}
assert ledger == Counter(A=7, B=0, C=-2)

positive, negative, zero_keys, ledger

(Counter({'A': 7}),
 Counter({'C': 2}),
 {'B'},
 Counter({'A': 7, 'B': 0, 'C': -2}))

## Problem 17 — Inventory transfer between warehouses

A transfer request is itself a Counter.

Write a function that:

1. validates that the source warehouse can satisfy the transfer,
2. returns **new** source and destination counters,
3. does not mutate the originals,
4. rejects negative transfer quantities.

In [30]:
def transfer_inventory(
    source: Mapping[T, int],
    destination: Mapping[T, int],
    transfer: Mapping[T, int],
) -> tuple[Counter[T], Counter[T]]:
    """Return new source/destination counters after a validated transfer."""
    source_c = Counter(source)
    destination_c = Counter(destination)
    transfer_c = Counter(transfer)

    if any(qty < 0 for qty in transfer_c.values()):
        raise ValueError("transfer quantities must be non-negative")

    transfer_c = +transfer_c

    if not can_fulfill(source_c, transfer_c):
        raise ValueError("source has insufficient inventory")

    new_source = source_c.copy()
    new_source.subtract(transfer_c)

    new_destination = destination_c.copy()
    new_destination.update(transfer_c)

    return new_source, new_destination


source = Counter(a=10, b=5)
destination = Counter(a=1, c=8)

new_source, new_destination = transfer_inventory(
    source,
    destination,
    {"a": 4, "b": 2},
)

assert new_source == Counter(a=6, b=3)
assert new_destination == Counter(c=8, a=5, b=2)
assert source == Counter(a=10, b=5)
assert destination == Counter(a=1, c=8)

new_source, new_destination

(Counter({'a': 6, 'b': 3}), Counter({'c': 8, 'a': 5, 'b': 2}))

## Problem 18 — Batch stream processing

A large stream cannot be stored entirely in memory.

Write `count_in_batches(stream, batch_size)` that:

- reads at most `batch_size` items into a temporary batch,
- updates a single Counter,
- validates `batch_size > 0`.

This still stores one entry per distinct key, but not the full raw stream.

In [31]:
def count_in_batches(stream: Iterable[T], batch_size: int = 1000) -> Counter[T]:
    """Count a stream using bounded raw-item batches."""
    if batch_size <= 0:
        raise ValueError("batch_size must be > 0")

    result: Counter[T] = Counter()
    batch: list[T] = []

    for item in stream:
        batch.append(item)

        if len(batch) == batch_size:
            result.update(batch)
            batch.clear()

    if batch:
        result.update(batch)

    return result


stream = (i % 7 for i in range(10_000))
counts = count_in_batches(stream, batch_size=128)

assert sum(counts.values()) == 10_000
assert set(counts) == set(range(7))

counts

Counter({0: 1429, 1: 1429, 2: 1429, 3: 1429, 4: 1428, 5: 1428, 6: 1428})

## Problem 19 — Find the least common positive items

`most_common()` ranks high-to-low. Create a helper for the `n` least common **positive-count** items with deterministic tie-breaking.

Sort by:
1. count ascending,
2. `repr(key)` ascending.

In [32]:
def least_common_positive(c: Counter[T], n: int) -> list[tuple[T, int]]:
    """Return n least-common positive-count items deterministically."""
    if n < 0:
        raise ValueError("n must be >= 0")

    positive_items = ((key, count) for key, count in c.items() if count > 0)

    return sorted(
        positive_items,
        key=lambda item: (item[1], repr(item[0])),
    )[:n]


c = Counter(a=5, b=1, c=1, d=3, e=0, f=-2)
result = least_common_positive(c, 3)

assert result == [("b", 1), ("c", 1), ("d", 3)]
result

[('b', 1), ('c', 1), ('d', 3)]

## Problem 20 — Reconstruct only when it is actually needed

Write `expand_inventory(counter, max_items)` that returns a list of repeated items using `elements()`, but refuses to materialize an excessively large result.

This makes the memory cost explicit.

In [33]:
def expand_inventory(c: Counter[T], max_items: int = 10_000) -> list[T]:
    """Materialize positive counts if the result is within max_items."""
    if max_items < 0:
        raise ValueError("max_items must be >= 0")

    positive = +c
    total_items = sum(positive.values())

    if total_items > max_items:
        raise ValueError(
            f"refusing to materialize {total_items} items; max_items={max_items}"
        )

    return list(positive.elements())


inventory = Counter(a=3, b=2, ignored=0, debt=-5)
expanded = expand_inventory(inventory, max_items=10)

assert Counter(expanded) == Counter(a=3, b=2)
assert len(expanded) == 5

expanded

['a', 'a', 'a', 'b', 'b']

## Problem 21 — Debugging weighted records

A common mistake is:

```python
Counter([("apple", 3), ("banana", 2)])
```

That counts each **tuple** once; it does not interpret the second tuple element as a weight.

Demonstrate the bug and fix it.

In [34]:
records = [("apple", 3), ("banana", 2), ("apple", 4)]

wrong = Counter(records)
correct = weighted_counter(records)

assert wrong[("apple", 3)] == 1
assert wrong["apple"] == 0

assert correct == Counter(apple=7, banana=2)

print("wrong  :", wrong)
print("correct:", correct)

wrong  : Counter({('apple', 3): 1, ('banana', 2): 1, ('apple', 4): 1})
correct: Counter({'apple': 7, 'banana': 2})


## Problem 22 — Implement a plain-dictionary equivalent

Re-implement the orders/refunds top-seller workflow without `Counter`.

Requirements:

- aggregate orders,
- subtract refunds,
- remove non-positive values,
- rank descending by quantity and alphabetically for ties,
- return top `k`.

Then compare it with a Counter-based solution.

In [35]:
def top_net_sales_dict(
    orders: Iterable[tuple[str, int]],
    refunds: Iterable[tuple[str, int]],
    k: int = 3,
) -> list[tuple[str, int]]:
    """Plain-dictionary implementation."""
    if k < 0:
        raise ValueError("k must be >= 0")

    net: dict[str, int] = {}

    for item, qty in orders:
        net[item] = net.get(item, 0) + qty

    for item, qty in refunds:
        net[item] = net.get(item, 0) - qty

    positive = {item: qty for item, qty in net.items() if qty > 0}

    return sorted(
        positive.items(),
        key=lambda item: (-item[1], item[0]),
    )[:k]


def top_net_sales_counter(
    orders: Iterable[tuple[str, int]],
    refunds: Iterable[tuple[str, int]],
    k: int = 3,
) -> list[tuple[str, int]]:
    """Counter-based implementation with the same tie-breaking rule."""
    net = weighted_counter(orders) - weighted_counter(refunds)

    return sorted(
        net.items(),
        key=lambda item: (-item[1], item[0]),
    )[:k]


orders = [
    ("battery", 10),
    ("mouse", 8),
    ("keyboard", 12),
    ("battery", 5),
    ("cable", 4),
]
refunds = [
    ("battery", 2),
    ("keyboard", 5),
    ("cable", 7),
]

dict_result = top_net_sales_dict(orders, refunds)
counter_result = top_net_sales_counter(orders, refunds)

assert dict_result == counter_result
dict_result

[('battery', 13), ('mouse', 8), ('keyboard', 7)]

## Problem 23 — Randomized equivalence test

Use deterministic pseudo-random data to compare the plain-dictionary and Counter implementations across many cases.

This is a lightweight property-testing style.

In [36]:
def randomized_equivalence_test(trials: int = 250, seed: int = 0) -> None:
    """Assert that dict and Counter top-net-sales implementations agree."""
    rng = random.Random(seed)
    products = ["battery", "charger", "cable", "case", "keyboard", "mouse"]

    for _ in range(trials):
        orders = [
            (rng.choice(products), rng.randint(1, 8))
            for _ in range(rng.randint(0, 40))
        ]

        refunds = [
            (rng.choice(products), rng.randint(1, 5))
            for _ in range(rng.randint(0, 20))
        ]

        k = rng.randint(0, len(products))

        assert top_net_sales_dict(orders, refunds, k) == top_net_sales_counter(
            orders,
            refunds,
            k,
        )


randomized_equivalence_test()
print("250 randomized equivalence trials passed.")

250 randomized equivalence trials passed.


## Problem 24 — End-to-end retail analysis

Given order and refund records, produce:

- sold quantities,
- refunded quantities,
- signed net quantities,
- positive net quantities,
- top 3 positive net sellers,
- products with net-negative quantities,
- total units sold,
- total units refunded.

Use one function returning a dictionary report.

In [37]:
def retail_report(
    orders: Iterable[tuple[str, int]],
    refunds: Iterable[tuple[str, int]],
    top_n: int = 3,
) -> dict[str, object]:
    """Build an end-to-end Counter-based retail report."""
    if top_n < 0:
        raise ValueError("top_n must be >= 0")

    sold = weighted_counter(orders)
    refunded = weighted_counter(refunds)

    signed_net = sold.copy()
    signed_net.subtract(refunded)

    positive_net = +signed_net
    negative_net = -signed_net

    top_sellers = sorted(
        positive_net.items(),
        key=lambda item: (-item[1], item[0]),
    )[:top_n]

    return {
        "sold": sold,
        "refunded": refunded,
        "signed_net": signed_net,
        "positive_net": positive_net,
        "negative_net": negative_net,
        "top_sellers": top_sellers,
        "total_sold": sum(sold.values()),
        "total_refunded": sum(refunded.values()),
    }


orders = [
    ("keyboard", 20),
    ("battery", 15),
    ("mouse", 12),
    ("keyboard", 6),
    ("cable", 5),
    ("charger", 4),
]

refunds = [
    ("keyboard", 3),
    ("battery", 2),
    ("mouse", 4),
    ("cable", 7),
]

report = retail_report(orders, refunds)

assert report["positive_net"] == Counter(
    keyboard=23,
    battery=13,
    mouse=8,
    charger=4,
)
assert report["negative_net"] == Counter(cable=2)
assert report["top_sellers"] == [
    ("keyboard", 23),
    ("battery", 13),
    ("mouse", 8),
]
assert report["total_sold"] == 62
assert report["total_refunded"] == 16

report

{'sold': Counter({'keyboard': 26,
          'battery': 15,
          'mouse': 12,
          'cable': 5,
          'charger': 4}),
 'refunded': Counter({'cable': 7, 'mouse': 4, 'keyboard': 3, 'battery': 2}),
 'signed_net': Counter({'keyboard': 23,
          'battery': 13,
          'mouse': 8,
          'charger': 4,
          'cable': -2}),
 'positive_net': Counter({'keyboard': 23,
          'battery': 13,
          'mouse': 8,
          'charger': 4}),
 'negative_net': Counter({'cable': 2}),
 'top_sellers': [('keyboard', 23), ('battery', 13), ('mouse', 8)],
 'total_sold': 62,
 'total_refunded': 16}

## Problem 25 — Counter-based document term profile

Build a normalized term-frequency Counter for each document, then answer:

1. which terms appear in **both** documents (`&`),
2. maximum term counts across either document (`|`),
3. terms/counts left when document A's counts are reduced by document B (`-`),
4. cosine similarity.

This combines several Counter techniques.

In [38]:
def term_profile(text: str) -> Counter[str]:
    """Return normalized term frequencies."""
    return Counter(tokenize_words(text))


doc_a = """
Counters are useful for counting data.
Data analysis often starts with counting.
"""

doc_b = """
Counting events with counters makes frequency analysis concise.
"""

a = term_profile(doc_a)
b = term_profile(doc_b)

shared = a & b
union_max = a | b
a_minus_b = a - b
similarity = counter_cosine_similarity(a, b)

assert shared["counting"] == 1
assert shared["counters"] == 1
assert 0 < similarity < 1

print("A          :", a)
print("B          :", b)
print("A & B      :", shared)
print("A | B      :", union_max)
print("A - B      :", a_minus_b)
print("cosine     :", similarity)

A          : Counter({'counting': 2, 'data': 2, 'counters': 1, 'are': 1, 'useful': 1, 'for': 1, 'analysis': 1, 'often': 1, 'starts': 1, 'with': 1})
B          : Counter({'counting': 1, 'events': 1, 'with': 1, 'counters': 1, 'makes': 1, 'frequency': 1, 'analysis': 1, 'concise': 1})
A & B      : Counter({'counters': 1, 'counting': 1, 'analysis': 1, 'with': 1})
A | B      : Counter({'counting': 2, 'data': 2, 'counters': 1, 'are': 1, 'useful': 1, 'for': 1, 'analysis': 1, 'often': 1, 'starts': 1, 'with': 1, 'events': 1, 'makes': 1, 'frequency': 1, 'concise': 1})
A - B      : Counter({'data': 2, 'are': 1, 'useful': 1, 'for': 1, 'counting': 1, 'often': 1, 'starts': 1})
cosine     : 0.44194173824159216


# Part III — Pitfalls and semantic edge cases

## Pitfall 1 — `subtract()` and `-` are not interchangeable

In [39]:
a = Counter(x=2)
b = Counter(x=5, y=1)

signed = a.copy()
signed.subtract(b)

positive_only = a - b

assert signed == Counter(x=-3, y=-1)
assert positive_only == Counter()

print("subtract():", signed)
print("a - b     :", positive_only)

subtract(): Counter({'y': -1, 'x': -3})
a - b     : Counter()


## Pitfall 2 — Zero-count keys may remain stored

A Counter can contain a key whose count is zero. Iterating stored keys is different from asking for the positive multiset content.

Unary `+` is a convenient way to create a positive-only cleaned Counter.

In [40]:
c = Counter(a=2, b=0, c=-1)

print("raw keys       :", list(c))
print("positive clean :", +c)

assert "b" in c
assert "b" not in +c

raw keys       : ['a', 'b', 'c']
positive clean : Counter({'a': 2})


## Pitfall 3 — `elements()` is not a general weighted-aggregation strategy

If a count is huge, materializing repeated elements can be wasteful. Prefer direct arithmetic on counts.

In [41]:
huge = Counter(widget=1_000_000)

assert huge["widget"] == 1_000_000

# Avoid this unless you truly need the million-element sequence:
# list(huge.elements())

## Pitfall 4 — Count values and multiset arithmetic

Counter can technically hold values other than positive integers, but several multiset-oriented operations are most natural with numeric counts, and `elements()` is specifically designed around integer repetition counts.

In [42]:
scores = Counter(alice=2.5, bob=1.5)

assert scores["nobody"] == 0
assert sum(scores.values()) == 4.0

scores

Counter({'alice': 2.5, 'bob': 1.5})

# Part IV — Mini challenge set

## Challenge A — First non-repeating character

Return the first character that occurs exactly once, or `None`.

In [43]:
def first_unique_character(text: str) -> str | None:
    """Return first character with frequency 1."""
    counts = Counter(text)
    return next((ch for ch in text if counts[ch] == 1), None)


assert first_unique_character("swiss") == "w"
assert first_unique_character("aabb") is None

## Challenge B — Bag equality

Two sequences represent the same bag/multiset when their item frequencies match, regardless of order.

In [44]:
def same_bag(left: Iterable[T], right: Iterable[T]) -> bool:
    """Return whether two iterables have identical multiplicities."""
    return Counter(left) == Counter(right)


assert same_bag([1, 2, 2, 3], [3, 2, 1, 2])
assert not same_bag([1, 2, 2], [1, 2])

## Challenge C — Excess items

Return what remains in `have` after fulfilling `need`, discarding zero/negative results.

In [45]:
def excess_items(have: Mapping[T, int], need: Mapping[T, int]) -> Counter[T]:
    """Return positive excess inventory."""
    return Counter(have) - Counter(need)


assert excess_items(
    {"a": 5, "b": 2},
    {"a": 3, "b": 2, "c": 1},
) == Counter(a=2)

## Challenge D — Common capacity

Two warehouses have inventories. Return the quantities both warehouses can guarantee simultaneously.

This is the elementwise minimum: `&`.

In [46]:
def common_capacity(a: Mapping[T, int], b: Mapping[T, int]) -> Counter[T]:
    """Return elementwise positive minima."""
    return Counter(a) & Counter(b)


assert common_capacity(
    {"a": 5, "b": 1},
    {"a": 2, "b": 7, "c": 3},
) == Counter(a=2, b=1)

## Challenge E — Peak combined capacity

Return elementwise maximum quantities using `|`.

In [47]:
def peak_capacity(a: Mapping[T, int], b: Mapping[T, int]) -> Counter[T]:
    """Return elementwise positive maxima."""
    return Counter(a) | Counter(b)


assert peak_capacity(
    {"a": 5, "b": 1},
    {"a": 2, "b": 7, "c": 3},
) == Counter(b=7, a=5, c=3)

# Part V — Complexity and design notes

Let:

- `n` = number of input items,
- `k` = number of distinct keys.

Typical costs:

- `Counter(iterable)`: approximately **O(n)** time, **O(k)** space.
- Single-key lookup/update: average **O(1)**.
- Iterating `.items()`: **O(k)**.
- Full deterministic sort: **O(k log k)**.
- `elements()`: produces work proportional to the sum of positive counts.
- Counter-to-Counter algebra: generally proportional to the number of involved distinct keys.

## Choosing the right operation

| Goal | Recommended operation |
|---|---|
| Count raw items | `Counter(iterable)` |
| Add another batch | `.update(...)` |
| Preserve signed subtraction | `.subtract(...)` |
| Positive difference only | `a - b` |
| Shared multiplicities | `a & b` |
| Maximum multiplicities | `a | b` |
| Remove non-positive counts | `+c` |
| Magnitudes of negative counts | `-c` |
| Top frequencies | `.most_common(n)` |
| Expand positive integer counts | `.elements()` |

# Part VI — Final mastery exercise

Build a reusable `InventoryAnalyzer` that supports:

- recording sales,
- recording refunds,
- signed net inventory movement,
- positive net movement,
- top positive movers,
- shortages/negative movers,
- merging another analyzer.

The implementation should avoid expanding weighted quantities.

In [48]:
class InventoryAnalyzer:
    """Track sold and refunded quantities with Counter-based aggregation."""

    def __init__(self) -> None:
        self.sold: Counter[str] = Counter()
        self.refunded: Counter[str] = Counter()

    def record_sale(self, item: str, quantity: int = 1) -> None:
        """Record a positive sale quantity."""
        if quantity < 0:
            raise ValueError("sale quantity must be non-negative")
        self.sold[item] += quantity

    def record_refund(self, item: str, quantity: int = 1) -> None:
        """Record a positive refund quantity."""
        if quantity < 0:
            raise ValueError("refund quantity must be non-negative")
        self.refunded[item] += quantity

    def signed_net(self) -> Counter[str]:
        """Return sold - refunded while preserving non-positive results."""
        result = self.sold.copy()
        result.subtract(self.refunded)
        return result

    def positive_net(self) -> Counter[str]:
        """Return positive net quantities only."""
        return +self.signed_net()

    def negative_net(self) -> Counter[str]:
        """Return magnitudes of negative net quantities."""
        return -self.signed_net()

    def top_movers(self, n: int = 3) -> list[tuple[str, int]]:
        """Return deterministic top positive movers."""
        if n < 0:
            raise ValueError("n must be >= 0")

        return sorted(
            self.positive_net().items(),
            key=lambda item: (-item[1], item[0]),
        )[:n]

    def merge(self, other: "InventoryAnalyzer") -> "InventoryAnalyzer":
        """Return a new analyzer combining two analyzers."""
        merged = InventoryAnalyzer()
        merged.sold = self.sold + other.sold
        merged.refunded = self.refunded + other.refunded
        return merged

In [49]:
north = InventoryAnalyzer()
north.record_sale("keyboard", 10)
north.record_sale("battery", 8)
north.record_refund("keyboard", 2)

south = InventoryAnalyzer()
south.record_sale("keyboard", 5)
south.record_sale("mouse", 9)
south.record_refund("battery", 3)

combined = north.merge(south)

assert combined.signed_net() == Counter(
    keyboard=13,
    mouse=9,
    battery=5,
)
assert combined.top_movers(2) == [
    ("keyboard", 13),
    ("mouse", 9),
]

combined.signed_net(), combined.top_movers(3)

(Counter({'keyboard': 13, 'mouse': 9, 'battery': 5}),
 [('keyboard', 13), ('mouse', 9), ('battery', 5)])

# Self-check summary

You should now be able to explain and implement:

- why `Counter` is more specialized than a plain dictionary/defaultdict for frequency work,
- how constructor behavior differs for iterables versus mappings,
- how missing counts behave,
- when `most_common` is enough and when deterministic sorting is better,
- how `elements()` reconstructs repeated positive counts,
- the semantic difference between `update`, `subtract`, `+`, and `-`,
- how `&` and `|` implement multiset min/max,
- how unary `+` and `-` help clean or separate signed counters,
- how to solve inventory, refund, text, sliding-window, and sparse-vector problems,
- how to avoid unnecessary materialization,
- how to test a Counter implementation against a plain-dictionary reference implementation.